# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** This notebook trains a learned model on the
same March 2026 feature frame as `w04_baseline_score.ipynb`, splits it honestly by client, and
checks whether it beats the Week-4 rule at picking pages to review first.

Setup matches `w03_data_contract.ipynb` and `w04_baseline_score.ipynb`: DuckDB reading remote
Parquet directly (see `skills/querying-big-datasets`), iterating on `month=2026-03` (mid-panel).
`month=2026-06` and the `_sample` table stay sealed — not touched anywhere in this notebook.

In [25]:
%pip install -q duckdb scikit-learn
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

FACT_MONTH  = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

print("DuckDB connected. Confirming the March partition is visible...")
print(con.sql(f"SELECT COUNT(*) AS rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {FACT_MONTH}").df())

DuckDB connected. Confirming the March partition is visible...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows      min_d      max_d
0  9841378 2026-03-01 2026-03-31


Schema probe

In [26]:
# Schema probe — run once if a later query errors on an unknown column
print("dim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT} LIMIT 1").df())

dim_content columns:
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         

## 1. Method choice and why

- **The question shape:** Lane 2 is "which pages should I review first?" — a ranking problem,
  not a plain yes/no. Per `skills/training-honest-models`, a ranking question needs a
  classifier's probability score, evaluated at precision@K — not a hard label.
- **My baseline is already a hand rule.** The point of this notebook is to see if a fitted model
  reads the same signals better than my fixed rule did.
- **Method: Logistic Regression first, then Random Forest.**
  - Logistic Regression: a coefficient I can read and explain to a non-technical stakeholder.
    If it already beats the baseline, I don't need anything heavier.
  - Random Forest: handles non-linear effects and feature interactions that a single hand-rule
    (`stale AND visible`) can't express — for example, staleness might only matter at certain
    traffic levels, which is exactly the "AND" my rule hard-coded rather than learned.
  - I stop at Random Forest. Gradient boosting is on the menu, but with roughly 30k rows and 8
    features it's unlikely to add real lift over a Random Forest — added complexity has to earn
    its place, not just look impressive on paper.
- **Same evaluation metric as the baseline:** Precision@50 on the ranked queue, computed the
  same way `w04_baseline_score.ipynb` computed it, so the comparison is apples-to-apples.
- **Same underlying data:** the same March 2026 feature frame as the baseline (one row per
  `content_hash_id x client_hash_id`, same `is_declining_proxy` label), with a few extra honest
  features added (GA4 engagement, word count) that the model — but not the fixed rule — can
  make use of.

In [27]:
# Rebuild the same feature frame as w04_baseline_score.ipynb, plus a few extra honest
# features the model can use that the fixed rule did not.
feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions)                                                            AS impressions_month,
        SUM(f.gsc_clicks)                                                                  AS clicks_month,
        AVG(NULLIF(f.gsc_avg_position, 0))                                                 AS avg_position_month,
        SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0) * 100                        AS ctr_month,
        SUM(CASE WHEN f.ga4_data_available THEN f.ga4_engaged_sessions ELSE 0 END)         AS engaged_sessions_month,
        SUM(CASE WHEN f.ga4_data_available THEN f.ga4_sessions ELSE 0 END)                 AS sessions_month,
        MAX(CASE WHEN f.ga4_data_available THEN 1 ELSE 0 END)                              AS has_ga4_month,
        DATE_DIFF('day', ANY_VALUE(c.content_created_date), DATE '2026-03-31')             AS content_age_days,
        ANY_VALUE(c.word_count)                                                            AS word_count,
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impr_last15,
        SUM(CASE WHEN f.report_date <  DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impr_prev15
    FROM {FACT_MONTH} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    GROUP BY f.content_hash_id, f.client_hash_id
    HAVING SUM(f.gsc_impressions) > 0
""").df()

# Drop rows with missing critical feature (content_age_days may be NULL if content_created_date is NULL)
feature_frame = feature_frame.dropna(subset=["content_age_days"]).reset_index(drop=True)

feature_frame["is_declining_proxy"] = (
    (feature_frame["impr_prev15"] > 0)
    & ((feature_frame["impr_last15"] - feature_frame["impr_prev15"]) / feature_frame["impr_prev15"] < -0.20)
).astype(int)

# GA4 engagement rate – 0 if no sessions; has_ga4_month already indicates tracking availability
feature_frame["engagement_rate_month"] = np.where(
    feature_frame["sessions_month"] > 0,
    feature_frame["engaged_sessions_month"] / feature_frame["sessions_month"] * 100,
    0.0,
)

# word_count missingness follows content_type – flag it instead of a blind fillna(0).
feature_frame["has_word_count"] = feature_frame["word_count"].notna().astype(int)
feature_frame["word_count"] = feature_frame["word_count"].fillna(0)

print(f"Rows: {len(feature_frame)}")
print(f"Base rate (is_declining_proxy): {feature_frame['is_declining_proxy'].mean():.1%}")
print("This row count and base rate should roughly match w04_baseline_score.ipynb's numbers (minus a few rows with missing age).")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738
Base rate (is_declining_proxy): 28.1%
This row count and base rate should roughly match w04_baseline_score.ipynb's numbers (minus a few rows with missing age).


,content_hash_id,client_hash_id,impressions_month,clicks_month,avg_position_month,ctr_month,engaged_sessions_month,sessions_month,has_ga4_month,content_age_days,word_count,impr_last15,impr_prev15,is_declining_proxy,engagement_rate_month,has_word_count
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,5.331238,0.000000,0.0,0.0,0,47,2999,70.0,111.0,1,0.0,1
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,34.0,0.0,6.419872,0.000000,0.0,0.0,0,47,3281,14.0,20.0,1,0.0,1
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,4.888929,0.000000,0.0,0.0,0,47,3579,20.0,57.0,1,0.0,1
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,5.177774,0.000000,0.0,0.0,0,47,2993,83.0,246.0,1,0.0,1
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,4.428747,0.664452,0.0,0.0,0,47,2455,403.0,199.0,0,0.0,1


## 2. Split design

- **Grouped by `client_hash_id`, not a random row split.** Pages from the same client share a
  template, a vertical, a content team, and a general traffic level. A random 70/30 split on
  rows would put some of a client's pages in train and others in test — the model could then
  "recognize" that client's pattern rather than learn a signal that generalizes to a *new*
  client's pages. `skills/flyrank/flyrank-data` says the same: client IDs are for
  grouping/splitting, never features, and grouped splits are the honest default here.
- **Not time-aware, on purpose.** This is one cross-sectional month (March 2026), not a
  forecast across months. The label itself already has its own internal time split
  (last-15-day vs. prior-15-day impressions) — that's inside the label, not the row split. A
  further time-based row split doesn't apply here; a client-based split is what stops leakage
  for this design.
- **Same dev month as the baseline:** `2026-03`. `2026-06` and `_sample` stay untouched, same
  rule as every notebook so far.
- **Split size:** 70% of clients for training, 30% held out for testing, `random_state=42` so
  the split is reproducible.

In [28]:
from sklearn.model_selection import GroupShuffleSplit

groups = feature_frame["client_hash_id"].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(gss.split(feature_frame, groups=groups))

train_df = feature_frame.iloc[train_idx].reset_index(drop=True)
test_df  = feature_frame.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])

print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"Train clients: {train_df['client_hash_id'].nunique()} | Test clients: {test_df['client_hash_id'].nunique()}")
print(f"Clients appearing in BOTH train and test (must be 0): {len(overlap)}")
print(f"Train base rate: {train_df['is_declining_proxy'].mean():.1%} | Test base rate: {test_df['is_declining_proxy'].mean():.1%}")

Train rows: 130904 | Test rows: 45834
Train clients: 32 | Test clients: 15
Clients appearing in BOTH train and test (must be 0): 0
Train base rate: 28.5% | Test base rate: 26.9%


**Verdict: CONFIRMED (split is honest)**

- Client overlap between train and test = **0** – the grouped split works.
- Train base rate = **28.5%**, test base rate = **26.9%**.
- Difference is **1.6 points**, which is small enough to trust the test numbers.
  The test set is slightly easier (fewer declining pages), but not enough to invalidate the comparison.

## 3. Train + compare vs my baseline

Same data, same metric, same split as your Week-4 baseline. Show the table.

The Week-4 rule (`score = stale x visible x impressions_month`) is recomputed here on the same
held-out test rows the models are scored on — not on the full March set like `w04` did. Scoring
the baseline on the full set and the model on a held-out slice would not be a fair comparison;
this way all three scores are read off the exact same rows.

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import json
import os

FEATURES = [
    "content_age_days", "impressions_month", "ctr_month", "avg_position_month",
    "engagement_rate_month", "has_ga4_month", "word_count", "has_word_count",
]

X_train = train_df[FEATURES].copy()
y_train = train_df["is_declining_proxy"].copy()
X_test = test_df[FEATURES].copy()
y_test = test_df["is_declining_proxy"].copy()

# Impute NaNs for avg_position_month and ctr_month with 0
X_train["avg_position_month"] = X_train["avg_position_month"].fillna(0)
X_test["avg_position_month"] = X_test["avg_position_month"].fillna(0)
X_train["ctr_month"] = X_train["ctr_month"].fillna(0)
X_test["ctr_month"] = X_test["ctr_month"].fillna(0)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

# --- Baseline rule, recomputed on the TEST split only (same rows as the model) ---
stale_te   = (test_df["content_age_days"] >= 180).astype(int)
visible_te = (test_df["impressions_month"] >= 500).astype(int)
baseline_score_te = (stale_te * visible_te * test_df["impressions_month"]).values

# --- Logistic Regression: readable, scaled inputs ---
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
logreg.fit(X_train, y_train)
logreg_score_te = logreg.predict_proba(X_test)[:, 1]

# --- Random Forest: can express the "stale AND visible" interaction on its own ---
rf = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=1
)
rf.fit(X_train, y_train)
rf_score_te = rf.predict_proba(X_test)[:, 1]

base_rate_te = float(y_test.mean())

rows = []
for name, scores in [
    ("Baseline rule (Week 4)", baseline_score_te),
    ("Logistic Regression",    logreg_score_te),
    ("Random Forest",          rf_score_te),
]:
    rows.append({
        "model": name,
        "precision_at_20": precision_at_k(scores, y_test.values, 20),
        "precision_at_50": precision_at_k(scores, y_test.values, 50),
    })

comparison = pd.DataFrame(rows)
print(f"Test-split base rate (for reference): {base_rate_te:.3f}")
print(comparison)

os.makedirs("work/outputs", exist_ok=True)
metrics_out = {
    "dev_month": "2026-03",
    "split": "GroupShuffleSplit by client_hash_id, test_size=0.30, random_state=42",
    "n_train": int(len(train_df)),
    "n_test": int(len(test_df)),
    "base_rate_test": base_rate_te,
    "comparison": comparison.to_dict(orient="records"),
    "features": FEATURES,
    "sklearn_note": "random_state=42 fixed throughout; tree-ensemble numbers can shift a few points across sklearn versions",
}
with open("work/outputs/w05_model_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
print("Wrote work/outputs/w05_model_metrics.json")

Test-split base rate (for reference): 0.269
                    model  precision_at_20  precision_at_50
0  Baseline rule (Week 4)             0.00             0.04
1     Logistic Regression             0.55             0.36
2           Random Forest             0.65             0.56
Wrote work/outputs/w05_model_metrics.json


**Verdict: FILLED**

Test base rate: **0.269**

| Model | Precision@20 | Precision@50 |
|---|---:|---:|
| Baseline rule (Week 4) | 0.00 | 0.04 |
| Logistic Regression | 0.55 | 0.36 |
| Random Forest | 0.65 | 0.56 |

- **Random Forest beats the Week‑4 rule by a large margin:** 0.56 vs 0.04 at Precision@50.
- **Logistic Regression also beats the baseline**, but Random Forest is stronger on both metrics.
- The ranking **does not flip** between @20 and @50 – Random Forest leads in both.
- Careful wording: **Random Forest beats the baseline on this held‑out client split.**

## 4. Errors and interpretation

Three things, in order: what the winning model leans on (permutation importance, not just
`.feature_importances_`, since permutation importance is checked by actually shuffling each
column rather than trusting the fit blindly), where it goes wrong by content age and traffic,
and three concrete wrong picks with a reason each.

In [30]:
from sklearn.inspection import permutation_importance

# Which model to inspect — set this after reading Section 3's table.
# If Logistic Regression wins, change to: model_to_inspect = logreg; model_score_te = logreg_score_te
model_to_inspect = rf
model_score_te = rf_score_te

perm = permutation_importance(
    model_to_inspect, X_test, y_test, n_repeats=20, random_state=42, scoring="average_precision"
)
importance_table = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)
print(importance_table)

# Where the model's own top-50 picks disagree with the proxy label
test_scored = test_df.copy()
test_scored["model_score"] = model_score_te
top50 = test_scored.sort_values("model_score", ascending=False).head(50)

false_positives = top50[top50["is_declining_proxy"] == 0]
print(f"\nFalse positives in the model's own top 50: {len(false_positives)} of 50")

age_bins = [0, 90, 180, 365, 730, np.inf]
age_labels = ["<90", "90-180", "180-365", "365-730", "730+"]
false_positives = false_positives.copy()
false_positives["age_bucket"] = pd.cut(false_positives["content_age_days"], bins=age_bins, labels=age_labels)
print(false_positives.groupby("age_bucket", observed=True).size())

# 3 concrete wrong cases: the highest-confidence false positives
wrong_cases = false_positives.sort_values("model_score", ascending=False).head(3)[
    ["content_hash_id", "model_score", "content_age_days", "impressions_month",
     "ctr_month", "avg_position_month", "engagement_rate_month", "is_declining_proxy"]
]
print("\nTop 3 wrong cases (highest model confidence, proxy label says not declining):")
print(wrong_cases)

# Leakage sanity check: the label's own ingredients must never be model features
print("\nOverlap between FEATURES and label-window columns (must be empty):",
      set(FEATURES) & {"impr_last15", "impr_prev15"})

                 feature  importance_mean  importance_std
0       content_age_days         0.065648        0.001696
1      impressions_month         0.035825        0.002752
2     avg_position_month         0.021106        0.001418
3          has_ga4_month         0.004031        0.001254
4         has_word_count         0.003896        0.000607
5              ctr_month         0.003442        0.001322
6             word_count         0.002473        0.000712
7  engagement_rate_month         0.000504        0.000254

False positives in the model's own top 50: 22 of 50
age_bucket
<90         5
180-365    17
dtype: int64

Top 3 wrong cases (highest model confidence, proxy label says not declining):
                content_hash_id  model_score  content_age_days  \
36740  content_b97542ab50e9264b     0.489468               228   
13840  content_2c2950faac96bf85     0.484763               228   
36759  content_b3803e892054d66e     0.483649               228   

       impressions_month  ctr

**Verdict: FILLED**

**Top 3 features by permutation importance:**
1. `content_age_days` (0.066) – older pages are more likely to be flagged as declining; matches the baseline’s staleness idea.
2. `impressions_month` (0.036) – traffic volume matters; visible pages are more likely to show decline signal.
3. `avg_position_month` (0.021) – lower positions (worse ranking) correlate with decline risk.

**False positives in the model’s own top 50:** **22 of 50** (44%).  
They are distributed across two age buckets: **5 are under 90 days old**, and **17 are between 180‑365 days old**. This tells us the model is confused by both very new pages (where there is little history) and middle‑aged pages (which may be stable despite their age).

**Top 3 wrong cases (model high confidence, label says not declining):**

1. `content_b97542ab50e9264b` – age 228 days, 1 impression, position 42, 0 CTR.  
   Model saw “old + low position” and predicted decline. Proxy label says not declining because there were **no prior‑period impressions** to decline from — this is a zero‑traffic page, not a declining one.

2. `content_2c2950faac96bf85` – age 228 days, 1 impression, position 9, 0 CTR.  
   Same pattern: old but essentially invisible. The model over‑weights age when traffic is near zero; a human reviewer would skip this page.

3. `content_b3803e892054d66e` – age 228 days, 1 impression, position 21, 0 CTR.  
   Again, a page with almost no traffic. The proxy label cannot mark it as declining because there is no baseline to decline from. This is a clear blind spot: **the model does not know how to treat old pages with almost no impressions.**

**Leakage check:** overlap printed an empty set `set()` – confirmed clean.

## Self-check

- [x] Section 2's client-overlap check prints 0 — the split is grouped, not leaking
- [x] Section 3's table has the baseline and every model, same test split, same metric, plus the base rate — computed in this notebook run, not copied from `w04`
- [x] Precision@20 was checked too – RF leads in both @20 (0.65) and @50 (0.56), so no disagreement.
- [x] Section 4 names the top 3 features and gives a plausible reason for each — not just a number
- [x] Section 4 shows 3 real wrong cases with real values, not hypothetical ones
- [x] The leakage check (label-window columns vs. FEATURES) printed an empty set
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.